# Team 1 data exploration, selection and packet formation

This notebook explores the source extract, records the 175-ticket draw, and forms the two packets that Ernest and YP labelled. Run it from the repository root or the `golden_set` folder; it reads `ict3113_tickets.csv` from the `golden_set` folder. That file is too large for the repository and is git-ignored, so place the course extract there before running. It draws from all Team 1 rows 1000-1999. It does not use earlier labels, resolve cases, or calculate agreement (see `analysis_and_agreement.ipynb`).

Run cells in order. The saved seed makes the draw repeatable, while write-once checks prevent a rerun from changing existing evidence. Once the packets contain answers, a rerun only verifies their row IDs and narratives; it never overwrites them. Outputs contain aggregate counts, never complaint narratives.

## 1. Read the source extract and derive Team 1 rows

Check the source schema and record counts before deriving Team 1's assigned integer row range, 1000 through 1999 inclusive.

In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import csv
import hashlib
import io
import json
import re
import secrets
import statistics
import sys

FIELDS = ("row", "source_label", "narrative")
CATEGORIES = (
    "Credit reporting", "Debt collection", "Mortgage", "Credit card",
    "Bank account or service", "Consumer loan", "Money transfer or service",
)
QUOTA = 25
METHOD = "sha256_rank_stratified_v1"
RANK_TEMPLATE = "ict3113-g1-fresh-v1:sample:{seed}:{source_label}:{row_id}"

def find_golden_set():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        for folder in (candidate, candidate / "golden_set"):
            if (folder / "ict3113_tickets.csv").is_file() and (folder / "protocol.md").is_file():
                return folder
    raise FileNotFoundError("Run from the repository root or golden_set folder; ict3113_tickets.csv must be in golden_set")

OUT = find_golden_set()
ARTIFACTS = OUT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
SOURCE = OUT / "ict3113_tickets.csv"

def read_csv(path, fields):
    field_limit = sys.maxsize
    while True:
        try:
            csv.field_size_limit(field_limit)
            break
        except OverflowError:
            field_limit //= 10
    with path.open("r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if tuple(reader.fieldnames or ()) != fields:
            raise ValueError(f"Unexpected schema in {path.name}: {reader.fieldnames}")
        records = list(reader)
    if any(None in row or set(row) != set(fields) for row in records):
        raise ValueError(f"Malformed CSV row in {path.name}")
    return records

source_rows = read_csv(SOURCE, FIELDS)
assert len(source_rows) == 50000, "Unexpected source record count"
assert all(row["row"] is not None and re.fullmatch(r"[0-9]+", row["row"]) for row in source_rows), "Non-integer source row ID"
team = [row for row in source_rows if 1000 <= int(row["row"]) <= 1999]
assert len(team) == 1000, "Team 1 must have exactly 1,000 rows"
print("Source records:", len(source_rows), "| Team 1 records:", len(team), "| row range: 1000-1999")

Source records: 50000 | Team 1 records: 1000 | row range: 1000-1999


## 2. Data Preparation

Before drawing, check every input column for null and blank values. Row IDs and narratives must each be unique; repeated source labels are expected because they define the seven sampling strata. A quality failure stops selection rather than quietly dropping or replacing records.

In [2]:
def quality_report(records):
    nulls = {field: sum(row.get(field) is None for row in records) for field in FIELDS}
    blanks = {
        field: sum(row.get(field) is not None and not row[field].strip() for row in records)
        for field in FIELDS
    }
    duplicate_ids = sum(count - 1 for count in Counter(row.get("row") for row in records).values() if count > 1)
    duplicate_narratives = sum(count - 1 for count in Counter(row.get("narrative") for row in records).values() if count > 1)
    duplicate_records = sum(
        count - 1 for count in Counter(tuple(row.get(field) for field in FIELDS) for row in records).values()
        if count > 1
    )
    counts = Counter(row.get("source_label") for row in records)
    unexpected = sorted(label for label in counts if label not in CATEGORIES)
    return {
        "null_counts": nulls,
        "blank_counts": blanks,
        "duplicate_row_id_occurrences": duplicate_ids,
        "duplicate_narrative_occurrences": duplicate_narratives,
        "duplicate_full_record_occurrences": duplicate_records,
        "source_counts": {label: counts[label] for label in CATEGORIES},
        "unexpected_source_labels": unexpected,
    }

prep = quality_report(team)
assert {int(row["row"]) for row in team} == set(range(1000, 2000)), "Missing or repeated Team 1 row ID"
assert not any(prep["null_counts"].values()) and not any(prep["blank_counts"].values()), "Null or blank Team 1 value"
assert prep["duplicate_row_id_occurrences"] == 0, "Duplicate Team 1 row ID"
assert prep["duplicate_narrative_occurrences"] == 0, "Duplicate Team 1 narrative"
assert prep["duplicate_full_record_occurrences"] == 0, "Duplicate Team 1 record"
assert not prep["unexpected_source_labels"], "Unexpected source category"
assert all(prep["source_counts"][label] >= QUOTA for label in CATEGORIES), "Insufficient source stratum"
print("Null values:", prep["null_counts"])
print("Blank values:", prep["blank_counts"])
print("Duplicate row IDs:", prep["duplicate_row_id_occurrences"],
      "| duplicate narratives:", prep["duplicate_narrative_occurrences"],
      "| duplicate full records:", prep["duplicate_full_record_occurrences"])
print("Available rows per noisy source category:")
for label in CATEGORIES:
    print(f"  {label}: {prep['source_counts'][label]}")

Null values: {'row': 0, 'source_label': 0, 'narrative': 0}
Blank values: {'row': 0, 'source_label': 0, 'narrative': 0}
Duplicate row IDs: 0 | duplicate narratives: 0 | duplicate full records: 0
Available rows per noisy source category:
  Credit reporting: 152
  Debt collection: 132
  Mortgage: 169
  Credit card: 136
  Bank account or service: 131
  Consumer loan: 151
  Money transfer or service: 129


In [3]:
lengths = defaultdict(list)
for row in team:
    lengths[row["source_label"]].append(len(row["narrative"].split()))
print("Narrative length in words per noisy source category (min / median / max):")
for label in CATEGORIES:
    words = lengths[label]
    print(f"  {label}: {min(words)} / {statistics.median(words):.0f} / {max(words)}")
all_words = [n for words in lengths.values() for n in words]
print(f"All Team 1 narratives: {min(all_words)} / {statistics.median(all_words):.0f} / {max(all_words)}")

Narrative length in words per noisy source category (min / median / max):
  Credit reporting: 28 / 118 / 359
  Debt collection: 38 / 128 / 368
  Mortgage: 39 / 173 / 351
  Credit card: 40 / 152 / 389
  Bank account or service: 32 / 147 / 371
  Consumer loan: 44 / 158 / 386
  Money transfer or service: 37 / 117 / 343
All Team 1 narratives: 28 / 140 / 389


## 3. Save the seed, then draw 25 from each noisy source category

The seven exact source-label strings define the strata. A fresh 128-bit seed is saved before ranking any real row. Within each stratum, sort by SHA-256 of the UTF-8 message shown below, breaking ties by numeric row ID, and take the first 25. This is a stratified deterministic pseudorandom draw, not an accuracy label. It balances raw source categories for coverage; the later human gold categories can have different counts. No prior ticket is excluded and no selected ticket is rerolled.

In [4]:
def rank_key(row, seed):
    row_id = int(row["row"])
    message = RANK_TEMPLATE.format(seed=seed, source_label=row["source_label"], row_id=row_id)
    return hashlib.sha256(message.encode("utf-8")).hexdigest(), row_id

def select_rows(records, seed):
    strata = defaultdict(list)
    for row in records:
        strata[row["source_label"]].append(row)
    selected_by_category = {}
    for label in CATEGORIES:
        if len(strata[label]) < QUOTA:
            raise ValueError(f"Fewer than {QUOTA} eligible rows in {label}")
        selected_by_category[label] = sorted(strata[label], key=lambda row: rank_key(row, seed))[:QUOTA]
    selected = sorted(
        (row for label in CATEGORIES for row in selected_by_category[label]),
        key=lambda row: int(row["row"]),
    )
    if len(selected) != 175 or len({row["row"] for row in selected}) != 175:
        raise AssertionError("Selection must contain 175 unique IDs")
    return selected, selected_by_category

SEED_PATH = OUT / "selection_seed.json"

def json_bytes(value):
    return (json.dumps(value, indent=2, ensure_ascii=False, sort_keys=True) + "\n").encode("utf-8")

def write_once(path, data):
    try:
        with path.open("xb") as stream:
            stream.write(data)
        return "created"
    except FileExistsError:
        if path.read_bytes() != data:
            raise FileExistsError(f"Existing evidence differs; refusing to overwrite {path.name}")
        return "verified"

if SEED_PATH.exists():
    seed_record = json.loads(SEED_PATH.read_text(encoding="utf-8"))
else:
    seed_record = {
        "method": METHOD,
        "rank_message_template": RANK_TEMPLATE,
        "seed_hex": secrets.token_hex(16),
        "team_row_range": [1000, 1999],
        "quota_per_source_label": QUOTA,
        "source_categories": list(CATEGORIES),
    }
    write_once(SEED_PATH, json_bytes(seed_record))
assert seed_record["method"] == METHOD
assert seed_record["rank_message_template"] == RANK_TEMPLATE
assert seed_record["team_row_range"] == [1000, 1999]
assert seed_record["quota_per_source_label"] == QUOTA
assert seed_record["source_categories"] == list(CATEGORIES)
seed = seed_record["seed_hex"]
assert isinstance(seed, str) and re.fullmatch(r"[0-9a-f]{32}", seed), "Invalid saved seed"
selected, selected_by_category = select_rows(team, seed)
print("Saved selection seed:", SEED_PATH.name)
print("Method:", METHOD, "| selected:", len(selected), "| per source category:", QUOTA)

Saved selection seed: selection_seed.json
Method: sha256_rank_stratified_v1 | selected: 175 | per source category: 25


## 4. Save a manifest of the draw

The manifest records the 175 selected row IDs, which both packets share. The method and seed are saved in `selection_seed.json`, so a reader with the course extract can reproduce the chosen IDs. The draw balances the noisy source categories (25 each); this is different from the balance of the final gold labels.

In [5]:
MANIFEST_FIELDS = ("row_id",)
PACKET_FIELDS = ("row_id", "narrative", "label", "uncertain", "reason_if_uncertain")

def csv_bytes(fields, records):
    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(buffer, fieldnames=fields, lineterminator="\n")
    writer.writeheader()
    writer.writerows(records)
    return buffer.getvalue().encode("utf-8")

manifest = [{"row_id": row["row"]} for row in selected]
manifest_data = csv_bytes(MANIFEST_FIELDS, manifest)
MANIFEST_PATH = ARTIFACTS / "sampling_manifest.csv"
print(MANIFEST_PATH.name, write_once(MANIFEST_PATH, manifest_data))
saved_manifest = read_csv(MANIFEST_PATH, MANIFEST_FIELDS)
assert saved_manifest == manifest
assert len({row["row_id"] for row in saved_manifest}) == 175
print("Selected counts by noisy source category:")
for label in CATEGORIES:
    print(f"  {label}: {len(selected_by_category[label])}")

sampling_manifest.csv verified
Selected counts by noisy source category:
  Credit reporting: 25
  Debt collection: 25
  Mortgage: 25
  Credit card: 25
  Bank account or service: 25
  Consumer loan: 25
  Money transfer or service: 25


## 5. Form and verify Ernest's and YP's packets

Each packet has the same 175 selected row IDs and narratives in numeric order, these packets will be sent to Ernest and Yp to review and comment. Only the five packet columns are present; source labels and earlier answers are omitted. A packet that does not exist yet is created with empty answer fields. A packet that already exists (and may now hold answers) is never replaced: the cell only checks that its columns, row IDs, order and narratives match the draw.

In [6]:
packet_rows = [
    {"row_id": row["row"], "narrative": row["narrative"],
     "label": "", "uncertain": "", "reason_if_uncertain": ""}
    for row in selected
]
packet_data = csv_bytes(PACKET_FIELDS, packet_rows)
for name in ("ernest-packet.csv", "yp-packet.csv"):
    path = OUT / name
    if path.exists():
        status = "present, not modified"
    else:
        path.write_bytes(packet_data)
        status = "created blank"
    saved = read_csv(path, PACKET_FIELDS)
    assert [row["row_id"] for row in saved] == [row["row_id"] for row in manifest]
    assert [row["narrative"] for row in saved] == [row["narrative"] for row in packet_rows]
    answered = sum(bool(row["label"]) for row in saved)
    print(f"{name}: {status} | rows: {len(saved)} | rows with a label: {answered}")
print("Packet formation complete. No narratives or source labels were printed.")

ernest-packet.csv: present, not modified | rows: 175 | rows with a label: 175
yp-packet.csv: present, not modified | rows: 175 | rows with a label: 175
Packet formation complete. No narratives or source labels were printed.
